# ==========================================
# FINANCIAL FRAUD DETECTION
# LOGISTIC REGRESSION MODEL
# ==========================================

# =========================
# 1. IMPORT LIBRARIES
# =========================

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# =========================
# 2. LOAD DATASET
# =========================

In [ ]:
df = pd.read_csv("PS_20174392719_1491204439457_log.csv")

print("Dataset Shape:")
print(df.shape)

df.head()

Dataset Shape:
(6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


# =========================
# 3. CHECK MISSING VALUES
# =========================

In [ ]:
print("\nMissing Values:")
print(df.isnull().sum())


Missing Values:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64


# =========================
# 4. KEEP FRAUD-RELATED TYPES
# =========================

In [ ]:
df = df[df["type"].isin(["TRANSFER", "CASH_OUT"])]
print("\nNew Shape:")
print(df.shape)


New Shape:
(2770409, 11)


# =========================
# 5. FEATURE ENGINEERING
# =========================

In [ ]:
df["errorBalanceOrig"] = (
    df["newbalanceOrig"]
    + df["amount"]
    - df["oldbalanceOrg"]
)

df["errorBalanceDest"] = (
    df["oldbalanceDest"]
    + df["amount"]
    - df["newbalanceDest"]
)

df["hourOfDay"] = df["step"] % 24

df["origEmptied"] = (
    df["newbalanceOrig"] == 0
).astype(int)

# =========================
# 6. CONVERT CATEGORICAL
# =========================

In [ ]:
df = pd.get_dummies(
    df,
    columns=["type"],
    drop_first=True
)


# =========================
# 7. DROP USELESS COLUMNS
# =========================

In [ ]:
df.drop(
    ["nameOrig", "nameDest"],
    axis=1,
    inplace=True
)

# =========================
# 8. FEATURES AND TARGET
# =========================

In [ ]:
X = df.drop("isFraud", axis=1)
y = df["isFraud"]
print("\nFeatures Shape:")
print(X.shape)


Features Shape:
(2770409, 12)


# =========================
# 9. TRAIN TEST SPLIT
# =========================

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("\nTraining Samples:", len(X_train))
print("Testing Samples:", len(X_test))


Training Samples: 2216327
Testing Samples: 554082


# =========================
# 10. SCALE FEATURES
# =========================

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =========================
# 11. TRAIN MODEL
# =========================


In [ ]:
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)
model.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

# =========================
# 12. PREDICT PROBABILITIES
# =========================

In [ ]:
probs = model.predict_proba(X_test_scaled)[:, 1]

# =========================
# 13. THRESHOLD TUNING
# =========================

In [ ]:
threshold = 0.99
pred = (probs > threshold).astype(int)

# =========================
# 14. EVALUATION
# =========================

In [ ]:
accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred)
recall = recall_score(y_test, pred)
f1 = f1_score(y_test, pred)
roc_auc = roc_auc_score(y_test, probs)
print("\n===== RESULTS =====")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("ROC AUC  :", roc_auc)
print("\nClassification Report\n")
print(classification_report(y_test, pred))


===== RESULTS =====
Accuracy : 0.9983919347677781
Precision: 0.7781065088757396
Recall   : 0.6402921485088253
F1 Score : 0.7025041736227045
ROC AUC  : 0.9960523216297642

Classification Report

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    552439
           1       0.78      0.64      0.70      1643

    accuracy                           1.00    554082
   macro avg       0.89      0.82      0.85    554082
weighted avg       1.00      1.00      1.00    554082



# =========================
# 15. CONFUSION MATRIX
# =========================

In [ ]:
cm = confusion_matrix(y_test, pred)
print("\nConfusion Matrix")
print(cm)


Confusion Matrix
[[552139    300]
 [   591   1052]]


# ===========================
# 16. FIND ROC-AUC AND PR-AUC
# ===========================

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.metrics import average_precision_score
pr_auc = average_precision_score(y_test, probs)
roc_auc = roc_auc_score(y_test, probs)
print("ROC-AUC:", roc_auc_score(y_test, probs))
print("PR-AUC :", average_precision_score(y_test, probs))

ROC-AUC: 0.9960523216297642
PR-AUC : 0.7870476222296117


# =========================
# 16. SAVE MODEL AND SCALER
# =========================

In [ ]:
os.makedirs("models", exist_ok=True)
joblib.dump(
    model,
    "models/logistic_regression.pkl"
)
print("\nModel Saved Successfully")
joblib.dump(
    scaler,
    "models/scaler.pkl"
)
print("Scaler Saved Successfully")


Model Saved Successfully
Scaler Saved Successfully
